# Lecture 7: Transformers and the Attention Mechanism


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

np.random.seed(42)
print("Libraries loaded.")


## 1. Tokenization and Embedding

Text → token → embedding vector

Each token is represented as a **vector** (embedding).  
These vectors are learned during training.


In [ ]:
# Simple word embedding simulation
vocab = ["the", "cat", "sat", "on", "mat", "dog", "ran", "over", "a", "fence"]
vocab_size = len(vocab)
d_model = 4  # embedding dimension (in practice 512, 768, 1024, etc.)

np.random.seed(7)
embedding_matrix = np.random.randn(vocab_size, d_model)

sentence = "the cat sat on the mat"
tokens = sentence.split()
token_ids = [vocab.index(t) for t in tokens]
embedded = embedding_matrix[token_ids]

print(f"Sentence: '{sentence}'")
print(f"Token IDs: {token_ids}")
print(f"Embedding matrix shape: {embedded.shape}")
print()
print("Token embedding vectors (first 3):")
for t, v in zip(tokens[:3], embedded[:3]):
    print(f"  '{t}': {v.round(3)}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
im = axes[0].imshow(embedding_matrix, cmap='RdBu_r', aspect='auto')
axes[0].set_title(f"Embedding Matrix ({vocab_size}x{d_model})")
axes[0].set_yticks(range(vocab_size)); axes[0].set_yticklabels(vocab)
axes[0].set_xlabel("Embedding dimension")
plt.colorbar(im, ax=axes[0])

im2 = axes[1].imshow(embedded, cmap='RdBu_r', aspect='auto')
axes[1].set_title(f"Token Embeddings in Sentence ({len(tokens)}x{d_model})")
axes[1].set_yticks(range(len(tokens))); axes[1].set_yticklabels(tokens)
axes[1].set_xlabel("Embedding dimension")
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.savefig("fig_nb07_embedding.png", dpi=100, bbox_inches='tight')
plt.show()


## 2. Positional Encoding

Transformers contain no inherent ordering information. Positional encoding provides it:

$$PE(pos, 2i) = \\sin\\left(\\frac{pos}{10000^{2i/d}}\\right)$$
$$PE(pos, 2i+1) = \\cos\\left(\\frac{pos}{10000^{2i/d}}\\right)$$


In [ ]:
def positional_encoding(seq_len, d_model):
    PE = np.zeros((seq_len, d_model))
    position = np.arange(seq_len)[:, None]
    div_term = np.exp(np.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
    PE[:, 0::2] = np.sin(position * div_term)
    PE[:, 1::2] = np.cos(position * div_term[:d_model//2])
    return PE

pe = positional_encoding(50, 64)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
im = axes[0].imshow(pe, cmap='RdBu_r', aspect='auto')
axes[0].set_title("Positional Encoding Matrix (50 positions, 64 dims)")
axes[0].set_xlabel("Embedding dimension (d)"); axes[0].set_ylabel("Position (token index)")
plt.colorbar(im, ax=axes[0])

for dim in [0, 1, 4, 8, 16]:
    axes[1].plot(pe[:, dim], label=f"dim={dim}")
axes[1].set_title("Positional Encoding: Different Dimensions")
axes[1].set_xlabel("Position"); axes[1].set_ylabel("Value")
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("fig_nb07_pe.png", dpi=100, bbox_inches='tight')
plt.show()


## 3. Scaled Dot-Product Self-Attention (Ch. 12.2)

For each token, Query (Q), Key (K), Value (V) vectors are computed:

$$\\text{Attention}(Q, K, V) = \\text{softmax}\\left(\\frac{QK^T}{\\sqrt{d_k}}\\right) V$$

- `Q·Kᵀ`: Similarity between each pair of tokens
- Scaling by `√d_k`: gradient stability
- Softmax: Normalized attention weights
- `× V`: Weighted sum of values


In [ ]:
def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    if mask is not None:
        scores = scores + mask * -1e9
    weights = softmax(scores)
    output = weights @ V
    return output, weights

# Demo: short sentence
np.random.seed(3)
seq_len = 6
d_k = 8
d_v = 8
d_model = 8

tokens_demo = ["I", "ate", "the", "green", "apple", "."]

# Dummy Q, K, V
X = np.random.randn(seq_len, d_model)
W_Q = np.random.randn(d_model, d_k) * 0.3
W_K = np.random.randn(d_model, d_k) * 0.3
W_V = np.random.randn(d_model, d_v) * 0.3

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

output, attn_weights = scaled_dot_product_attention(Q, K, V)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

im = axes[0].imshow(attn_weights, cmap='Blues', vmin=0, vmax=1)
axes[0].set_xticks(range(seq_len)); axes[0].set_xticklabels(tokens_demo, fontsize=10)
axes[0].set_yticks(range(seq_len)); axes[0].set_yticklabels(tokens_demo, fontsize=10)
axes[0].set_title("Self-Attention Weights\n(Row = query, Column = key)")
plt.colorbar(im, ax=axes[0])

for i in range(seq_len):
    for j in range(seq_len):
        axes[0].text(j, i, f"{attn_weights[i,j]:.2f}", ha='center', va='center',
                    fontsize=8, color='black' if attn_weights[i,j] < 0.5 else 'white')

im2 = axes[1].imshow(output, cmap='RdBu_r', aspect='auto')
axes[1].set_yticks(range(seq_len)); axes[1].set_yticklabels(tokens_demo)
axes[1].set_title("Attention Output: Context Vectors")
axes[1].set_xlabel("Dimension")
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.savefig("fig_nb07_attention.png", dpi=100, bbox_inches='tight')
plt.show()


## 4. Multi-Head Attention (Ch. 12.3)

Multiple attention heads run in parallel → different types of relationships are learned:

$$\\text{MultiHead}(Q,K,V) = \\text{Concat}(\\text{head}_1, \\ldots, \\text{head}_h) W^O$$
$$\\text{head}_i = \\text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$


In [ ]:
def multi_head_attention(X, n_heads=4, d_k=4, d_v=4):
    d_model = X.shape[1]
    seq_len = X.shape[0]
    all_outputs = []
    all_weights = []
    
    np.random.seed(42)
    for h in range(n_heads):
        W_Q = np.random.randn(d_model, d_k) * 0.3
        W_K = np.random.randn(d_model, d_k) * 0.3
        W_V = np.random.randn(d_model, d_v) * 0.3
        Q_h = X @ W_Q
        K_h = X @ W_K
        V_h = X @ W_V
        out_h, w_h = scaled_dot_product_attention(Q_h, K_h, V_h)
        all_outputs.append(out_h)
        all_weights.append(w_h)
    
    concat_output = np.concatenate(all_outputs, axis=-1)
    W_O = np.random.randn(n_heads*d_v, d_model) * 0.3
    return concat_output @ W_O, all_weights

d_model_demo = 16
X_demo = np.random.randn(6, d_model_demo)
mha_output, head_weights = multi_head_attention(X_demo, n_heads=4, d_k=4, d_v=4)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, (ax, weights) in enumerate(zip(axes, head_weights)):
    im = ax.imshow(weights, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f"Head {i+1}")
    ax.set_xticks(range(6)); ax.set_xticklabels(tokens_demo, fontsize=8, rotation=45)
    ax.set_yticks(range(6)); ax.set_yticklabels(tokens_demo, fontsize=8)
    plt.colorbar(im, ax=ax)

plt.suptitle("Multi-Head Attention: Attention Weights of 4 Heads", fontsize=13)
plt.tight_layout()
plt.savefig("fig_nb07_multihead.png", dpi=100, bbox_inches='tight')
plt.show()


## 5. Causal (Masked) Attention — GPT Style

In language models, each token can only attend to **past** tokens (future tokens are masked).


In [ ]:
# Causal mask
seq_len_c = 6
causal_mask = np.triu(np.ones((seq_len_c, seq_len_c)), k=1)  # 1=mask out

X_c = np.random.randn(seq_len_c, d_k)
Q_c, K_c, V_c = X_c, X_c, X_c

_, causal_weights = scaled_dot_product_attention(Q_c, K_c, V_c, mask=causal_mask)
_, full_weights   = scaled_dot_product_attention(Q_c, K_c, V_c, mask=None)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (name, w) in zip(axes, [("Full Attention (BERT-style)", full_weights),
                                  ("Causal Attention (GPT-style)", causal_weights)]):
    im = ax.imshow(w, cmap='Blues', vmin=0, vmax=1)
    for i in range(seq_len_c):
        for j in range(seq_len_c):
            ax.text(j, i, f"{w[i,j]:.2f}", ha='center', va='center', fontsize=9)
    ax.set_title(name)
    ax.set_xlabel("Key position (past →)"); ax.set_ylabel("Query position")
    plt.colorbar(im, ax=ax)

plt.suptitle("Encoder (full) vs Decoder (causal) Self-Attention", fontsize=13)
plt.tight_layout()
plt.savefig("fig_nb07_causal.png", dpi=100, bbox_inches='tight')
plt.show()


## 6. Transformer Block

A complete Transformer encoder block:

```
x → LayerNorm → MultiHeadAttention → + (skip) → LayerNorm → FFN → + (skip) → output
```


In [ ]:
def layer_norm(x, eps=1e-6):
    mean = x.mean(axis=-1, keepdims=True)
    std  = x.std(axis=-1, keepdims=True)
    return (x - mean) / (std + eps)

def ffn(x, d_ff=32):
    d_model = x.shape[-1]
    np.random.seed(5)
    W1 = np.random.randn(d_model, d_ff) * 0.1
    b1 = np.zeros(d_ff)
    W2 = np.random.randn(d_ff, d_model) * 0.1
    b2 = np.zeros(d_model)
    return np.maximum(0, x @ W1 + b1) @ W2 + b2

def transformer_block(X):
    seq_len, d_model = X.shape
    n_heads = 4
    d_k = d_model // n_heads
    
    # 1. Sub-layer: Self-attention
    X_norm1 = layer_norm(X)
    attn_out, _ = multi_head_attention(X_norm1, n_heads=n_heads, d_k=d_k, d_v=d_k)
    X = X + attn_out  # Residual
    
    # 2. Sub-layer: FFN
    X_norm2 = layer_norm(X)
    ff_out = ffn(X_norm2, d_ff=d_model*4)
    X = X + ff_out  # Residual
    
    return X

d_model_blk = 16
X_blk = np.random.randn(6, d_model_blk)
X_out = transformer_block(X_blk)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(X_blk, cmap='RdBu_r', aspect='auto')
axes[0].set_title("Transformer Block Input")
axes[0].set_yticks(range(6)); axes[0].set_yticklabels(tokens_demo)
axes[0].set_xlabel("d_model")

axes[1].imshow(X_out, cmap='RdBu_r', aspect='auto')
axes[1].set_title("Transformer Block Output (same shape)")
axes[1].set_yticks(range(6)); axes[1].set_yticklabels(tokens_demo)
axes[1].set_xlabel("d_model")

plt.suptitle("Transformer Block: Input -> Output (shape preserved)", fontsize=13)
plt.tight_layout()
plt.savefig("fig_nb07_block.png", dpi=100, bbox_inches='tight')
plt.show()


## 7. BERT vs GPT Comparison

| Feature | BERT (Encoder) | GPT (Decoder) |
|---|---|---|
| Attention | Full (bidirectional) | Causal (unidirectional) |
| Pre-training | Masked LM | Causal LM |
| Usage | Understanding tasks | Generation tasks |
| Example task | Classification, NER | Text generation, Chat |

## 8. Summary

| Concept | Description |
|---|---|
| **Self-Attention** | How strongly is each token related to the others? |
| **Q, K, V** | Query, Key, Value — the 3 components of attention computation |
| **Multi-head** | Different heads learn different relationship types |
| **Causal mask** | Hide future tokens → language model training |
| **Positional Encoding** | Add sequence order information to the embeddings |
| **Transformer block** | Attention + FFN + Skip + Norm |

**Next Notebook →** Generative Models (GAN, VAE, Diffusion)
